**Step 1: I check what are the most frequent diagnosis and medication based on the contact of pasient , display the top 20 then i categorize them in `Axis 1` category and display the distribution of category**

In [ ]:
import os
import sys

parent_dir = os.path.join(os.path.dirname(os.path.abspath("")))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from importall import *
from updated_mapped_icd_codes import mapped_icd_codes

output_plot = os.getenv("OUTPUT_PLOTS")
with open(os.getenv("ATCNAME_CODE_NORSK") + "atcnavn_code.json", "r") as f:
    atc_mapping = json.load(f)
atc_map = {code: info.get("atcnavn", "Unknown") for code, info in atc_mapping.items()}
input_path = os.getenv("AXISI_MAIN_DATASET")
df = dd.read_parquet(input_path, engine="pyarrow")

print(
    "Axis 1 Unique patients • all of the above + medicated:",
    df["pasient_nr"].nunique().compute(),
)
print(
    "Unique episode of care • all of the above + medicated:",
    df["sak_nr"].nunique().compute(),
)
print(
    "Unique contacts • all of the above + medicated:",
    df["opphold_id"].nunique().compute(),
)

all_diag_codes = df["diag_diagnose"].dropna().unique().compute()
all_diag_codes = sorted(all_diag_codes)
print("All diagnosis codes in Axis 1:")
for code in all_diag_codes:
    print(f"  {code}")

diag_counts = (
    df.groupby("diag_diagnose")["opphold_id"]
    .nunique()
    .compute()
    .sort_values(ascending=False)
    .head(20)
)
diag_labels = [
    f"{code} – {mapped_icd_codes.get(code, 'Unknown')}" for code in diag_counts.index
]
wrapped_diag_labels = [textwrap.fill(lbl, width=30) for lbl in diag_labels]

width = 0.6
x_diag = np.arange(len(diag_counts))
fig, ax = plt.subplots(figsize=(16, 8))
bars = ax.bar(x_diag, diag_counts.values, width=width, color="#0072B2", alpha=0.6)
ax.set_xticks(x_diag)
ax.set_xticklabels(wrapped_diag_labels, rotation=90)
ax.set_xlabel("Diagnosis (ICD-10 code – description)")
ax.set_ylabel("Unique Opphold ID Count")
ax.set_title("Axis 1 – Top 20 Primary Diagnoses by Opphold ID")
ax.bar_label(bars, labels=[f"{v:,}" for v in diag_counts.values], padding=2, fontsize=8)
plt.tight_layout()
plt.show()

med_counts = (
    df.groupby("atckode")["opphold_id"]
    .nunique()
    .compute()
    .sort_values(ascending=False)
    .head(20)
)

med_labels = [f"{code} – {atc_map.get(code, 'Unknown')}" for code in med_counts.index]
wrapped_med_labels = [textwrap.fill(lbl, width=30) for lbl in med_labels]

x_med = np.arange(len(med_counts))
fig, ax = plt.subplots(figsize=(16, 8))
bars = ax.bar(x_med, med_counts.values, width=width, color="#D55E00", alpha=0.6)
ax.set_xticks(x_med)
ax.set_xticklabels(wrapped_med_labels, rotation=90)
ax.set_xlabel("Medication (ATC code – name)")
ax.set_ylabel("Unique Opphold ID Count")
ax.set_title("Axis 1 – Top 20 Medications by Opphold ID")
ax.bar_label(bars, labels=[f"{v:,}" for v in med_counts.values], padding=2, fontsize=8)
plt.tight_layout()
plt.show()

categories = {
    "F03–F09\nOrganic, symptomatic": lambda c: "F03" <= c[:3] <= "F09",
    "F10–F19\nPsychoactive substances": lambda c: "F10" <= c[:3] <= "F19",
    "F20–F29\nSchizophrenia & paranoias": lambda c: "F20" <= c[:3] <= "F29",
    "F30–F39\nAffective/mood disorders": lambda c: "F30" <= c[:3] <= "F39",
    "F40–F48\nNeurotic & somatoform": lambda c: "F40" <= c[:3] <= "F48",
    "F50–F59\nBehavioral syndromes": lambda c: "F50" <= c[:3] <= "F59",
    "F60–F69\nPersonality disorders": lambda c: "F60" <= c[:3] <= "F69",
    "F84\nPervasive developmental": lambda c: c[:3] == "F84",
    "F90–F98\nChildhood emotional": lambda c: "F90" <= c[:3] <= "F98",
    "F99\nUnspecified mental disorder": lambda c: c[:3] == "F99",
    "R40–R46\nCognition & emotional signs": lambda c: "R40" <= c[:3] <= "R46",
    "Z00–Z71\nHealth services contacts": lambda c: "Z00" <= c[:3] <= "Z71",
}


def map_diag_category(code: str) -> str:
    if not isinstance(code, str):
        return "Other/Unclassified"
    for cat_name, cond in categories.items():
        try:
            if cond(code):
                return cat_name
        except Exception:
            continue
    return "Other/Unclassified"


_df_diag_cat = df[["diag_diagnose", "opphold_id"]].dropna(subset=["diag_diagnose"])
_df_diag_cat = _df_diag_cat.assign(
    diag_cat=_df_diag_cat["diag_diagnose"].map(
        map_diag_category, meta=("diag_cat", "object")
    )
)

diag_cat_counts = (
    _df_diag_cat.groupby("diag_cat")["opphold_id"]
    .nunique()
    .compute()
    .sort_values(ascending=False)
)

print("\nDiagnosis category → unique opphold_id counts:")
for cat, cnt in diag_cat_counts.items():
    print(f"  {cat}: {cnt:,}")

diag_cat_labels = [textwrap.fill(name, width=25) for name in diag_cat_counts.index]
x_dc = np.arange(len(diag_cat_counts))
fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.bar(x_dc, diag_cat_counts.values, width=0.8, color="grey", alpha=0.7)
ax.set_xticks(x_dc)
ax.set_xticklabels(diag_cat_labels, rotation=90)
ax.set_xlabel("Diagnosis Category")
ax.set_ylabel("Unique Opphold ID Count")
ax.bar_label(
    bars,
    labels=[f"{v:,} ({v/7179*100:.2f}%)" for v in diag_cat_counts.values],
    padding=2,
    fontsize=8,
)
plt.tight_layout()
plt.savefig(output_plot + "axis1/axis1_1(b)part1.png", dpi=400, bbox_inches="tight")
plt.show()

med_letter_map = {
    "A": "A-Alimentary tract & metabolism",
    "B": "B-Blood & blood forming organs",
    "C": "C-Cardiovascular system",
    "D": "D-Dermatologicals",
    "G": "G-Genito-urinary system & sex hormones",
    "H": "H-Systemic hormonal preparations (excl. sex hormones)",
    "J": "J-Anti-infectives for systemic use",
    "M": "M-Musculo-skeletal system",
    "N": "Other nervous system (N01-N02,N04,N07)",
    "R": "R-Respiratory system",
    "S": "S-Sensory organs",
    "V": "V-Various",
}


def map_med_category(code: str) -> str:
    if not isinstance(code, str) or len(code) < 1:
        return "Other/Unclassified"
    if code.startswith("N05A"):
        return "N05A-Antipsychotics nervous system"
    if code.startswith("N05B"):
        return "N05B-Anxiolytics nervous system"
    if code.startswith("N05C"):
        return "N05C-Hypnotics and sedatives nervous system"
    if code.startswith("N06A"):
        return "N06A-Antidepressants nervous system"
    if code.startswith("N06B"):
        return (
            "N06B-Psychostimulants, agents used for ADHD and nootropics nervous system"
        )
    if code.startswith("N06C"):
        return "N06C-Pscyholeptics and psychoanaleptics in combination nervous system"
    if code.startswith("N06D"):
        return "N06D-Anti-dementia drugs nervous system"
    if code.startswith("N03"):
        return "N03-Antiepileptics nervous system"

    letter = code[0]
    return med_letter_map.get(letter, "Other/Unclassified")


_df_med_cat = df[["atckode", "opphold_id"]].dropna(subset=["atckode"])
_df_med_cat = _df_med_cat.assign(
    med_cat=_df_med_cat["atckode"].map(map_med_category, meta=("med_cat", "object"))
)

med_cat_counts = (
    _df_med_cat.groupby("med_cat")["opphold_id"]
    .nunique()
    .compute()
    .sort_values(ascending=False)
)

print("\nMedication category → unique opphold_id counts:")
for cat, cnt in med_cat_counts.items():
    print(f"  {cat}: {cnt:,}")

special_cats = [
    "N05A-Antipsychotics nervous system",
    "N05B-Anxiolytics nervous system",
    "N05C-Hypnotics and sedatives nervous system",
    "N06A-Antidepressants nervous system",
    "N06B-Psychostimulants, agents used for ADHD and nootropics nervous system",
    "N06C-Pscyholeptics and psychoanaleptics in combination nervous system",
    "N06D-Anti-dementia drugs nervous system",
    "N03-Antiepileptics nervous system",
]

def_hex_default = "lightgrey"
highlight_color = "grey"

colors = [
    highlight_color if cat in special_cats else def_hex_default
    for cat in med_cat_counts.index
]

med_cat_labels = [textwrap.fill(name, width=25) for name in med_cat_counts.index]
x_mc = np.arange(len(med_cat_counts))

fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.bar(x_mc, med_cat_counts.values, width=0.8, color=colors)
ax.set_xticks(x_mc)
ax.set_xticklabels(med_cat_labels, rotation=90)
ax.set_xlabel("Medication Category (ATC Group)")
ax.set_ylabel("Unique Opphold ID Count")
ax.bar_label(
    bars,
    labels=[f"{v:,} ({v/7179*100:.2f}%)" for v in med_cat_counts.values],
    padding=2,
    fontsize=8,
)
plt.tight_layout()
plt.savefig(output_plot + "axis1/axis1_1(b)part2.png", dpi=400, bbox_inches="tight")
plt.show()

**Step 2: I check what are the most frequent diagnosis and medication based on the contact of pasient , display the top 20 then i categorize them in `Axis 2` category and display the distribution of category, with correct ATC category and color**

In [ ]:
import os
import sys

parent_dir = os.path.join(os.path.dirname(os.path.abspath("")))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from importall import *
from updated_mapped_icd_codes import mapped_icd_codes

output_plot = os.getenv("OUTPUT_PLOTS")

with open(os.getenv("ATCNAME_CODE_NORSK") + "atcnavn_code.json", "r") as f:
    atc_mapping = json.load(f)
atc_map = {code: info.get("atcnavn", "Unknown") for code, info in atc_mapping.items()}


input_path = os.getenv("AXISII_MAIN_DATASET")
df = dd.read_parquet(input_path, engine="pyarrow")
print(
    "Axis 1 Unique patients • all of the above + medicated:",
    df["pasient_nr"].nunique().compute(),
)
print(
    "Unique episode of care • all of the above + medicated:",
    df["sak_nr"].nunique().compute(),
)
print(
    "Unique contacts • all of the above + medicated:",
    df["opphold_id"].nunique().compute(),
)
all_diag_codes = df["diag_diagnose"].dropna().unique().compute()
all_diag_codes = sorted(all_diag_codes)
print("All diagnosis codes in Axis 1:")
for code in all_diag_codes:
    print(f"  {code}")

diag_counts = (
    df.groupby("diag_diagnose")["opphold_id"]
    .nunique()
    .compute()
    .sort_values(ascending=False)
    .head(20)
)

diag_labels = [
    f"{code} – {mapped_icd_codes.get(code, 'Unknown')}" for code in diag_counts.index
]
wrapped_diag_labels = [textwrap.fill(lbl, width=30) for lbl in diag_labels]
width = 0.6
x_diag = np.arange(len(diag_counts))
fig, ax = plt.subplots(figsize=(16, 8))
bars = ax.bar(x_diag, diag_counts.values, width=width, color="grey", alpha=0.7)
ax.set_xticks(x_diag)
ax.set_xticklabels(wrapped_diag_labels, rotation=90)
ax.set_xlabel("Diagnosis (ICD-10 code – description)")
ax.set_ylabel("Unique Opphold ID Count")
ax.set_title("Axis 2 – Top 20 Primary Diagnoses by Opphold ID")
ax.bar_label(bars, labels=[f"{v:,}" for v in diag_counts.values], padding=2, fontsize=8)
plt.tight_layout()
plt.show()

med_counts = (
    df.groupby("atckode")["opphold_id"]
    .nunique()
    .compute()
    .sort_values(ascending=False)
    .head(20)
)

med_labels = [f"{code} – {atc_map.get(code, 'Unknown')}" for code in med_counts.index]
wrapped_med_labels = [textwrap.fill(lbl, width=30) for lbl in med_labels]
x_med = np.arange(len(med_counts))
fig, ax = plt.subplots(figsize=(16, 8))
bars = ax.bar(x_med, med_counts.values, width=width, color="#D55E00", alpha=0.6)
ax.set_xticks(x_med)
ax.set_xticklabels(wrapped_med_labels, rotation=90)
ax.set_xlabel("Medication (ATC code – name)")
ax.set_ylabel("Unique Opphold ID Count")
ax.set_title("Axis 2 – Top 20 Medications by Opphold ID")
ax.bar_label(bars, labels=[f"{v:,}" for v in med_counts.values], padding=2, fontsize=8)
plt.tight_layout()
plt.show()

categories = {
    "F80\nSpeech & language": lambda c: c.replace(".", "")[:3] == "F80",
    "F81\nSchool skills & learning": lambda c: c.replace(".", "")[:3] == "F81",
    "F82\nMotor skills": lambda c: c.replace(".", "")[:3] == "F82",
    "F83\nMixed specific skills": lambda c: c.replace(".", "")[:3] == "F83",
    "F88\nOther psychological dev": lambda c: c.replace(".", "")[:3] == "F88",
    "F89\nUnspecified psych dev": lambda c: c.replace(".", "")[:3] == "F89",
}


def map_diag_category(code: str) -> str:
    if not isinstance(code, str):
        return "Other/Unclassified"
    for cat_name, cond in categories.items():
        try:
            if cond(code):
                return cat_name
        except Exception:
            continue
    return "Other/Unclassified"


_df_diag_cat = df[["diag_diagnose", "opphold_id"]].dropna(subset=["diag_diagnose"])
_df_diag_cat = _df_diag_cat.assign(
    diag_cat=_df_diag_cat["diag_diagnose"].map(
        map_diag_category, meta=("diag_cat", "object")
    )
)

diag_cat_counts = (
    _df_diag_cat.groupby("diag_cat")["opphold_id"]
    .nunique()
    .compute()
    .sort_values(ascending=False)
)

print("\nDiagnosis category → unique opphold_id counts:")
for cat, cnt in diag_cat_counts.items():
    print(f"  {cat}: {cnt:,}")

diag_cat_labels = [textwrap.fill(name, width=25) for name in diag_cat_counts.index]
x_dc = np.arange(len(diag_cat_counts))
fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.bar(x_dc, diag_cat_counts.values, width=0.8, color="grey", alpha=0.7)
ax.set_xticks(x_dc)
ax.set_xticklabels(diag_cat_labels, rotation=90)
ax.set_xlabel("Diagnosis Category")
ax.set_ylabel("Unique Opphold ID Count")
ax.bar_label(
    bars,
    labels=[f"{v:,} ({v/821*100:.2f}%)" for v in diag_cat_counts.values],
    padding=2,
    fontsize=8,
)
plt.tight_layout()
plt.savefig(output_plot + "axis2/axis2_1(b)part1.png", dpi=400, bbox_inches="tight")
plt.show()

med_letter_map = {
    "A": "A-Alimentary tract & metabolism",
    "B": "B-Blood & blood forming organs",
    "C": "C-Cardiovascular system",
    "D": "D-Dermatologicals",
    "G": "G-Genito-urinary system & sex hormones",
    "H": "H-Systemic hormonal preparations (excl. sex hormones)",
    "J": "J-Anti-infectives for systemic use",
    "M": "M-Musculo-skeletal system",
    "N": "Other nervous system (N01-N02,N04,N07)",
    "R": "R-Respiratory system",
    "S": "S-Sensory organs",
    "V": "V-Various",
}


def map_med_category(code: str) -> str:
    if not isinstance(code, str) or len(code) < 1:
        return "Other/Unclassified"
    if code.startswith("N05A"):
        return "N05A-Antipsychotics nervous system"
    if code.startswith("N05B"):
        return "N05B-Anxiolytics nervous system"
    if code.startswith("N05C"):
        return "N05C-Hypnotics and sedatives nervous system"
    if code.startswith("N06A"):
        return "N06A-Antidepressants nervous system"
    if code.startswith("N06B"):
        return (
            "N06B-Psychostimulants, agents used for ADHD and nootropics nervous system"
        )
    if code.startswith("N06C"):
        return "N06C-Pscyholeptics and psychoanaleptics in combination nervous system"
    if code.startswith("N06D"):
        return "N06D-Anti-dementia drugs nervous system"
    if code.startswith("N03"):
        return "N03-Antiepileptics nervous system"

    letter = code[0]
    return med_letter_map.get(letter, "Other/Unclassified")


_df_med_cat = df[["atckode", "opphold_id"]].dropna(subset=["atckode"])
_df_med_cat = _df_med_cat.assign(
    med_cat=_df_med_cat["atckode"].map(map_med_category, meta=("med_cat", "object"))
)

med_cat_counts = (
    _df_med_cat.groupby("med_cat")["opphold_id"]
    .nunique()
    .compute()
    .sort_values(ascending=False)
)

print("\nMedication category → unique opphold_id counts:")
for cat, cnt in med_cat_counts.items():
    print(f"  {cat}: {cnt:,}")

special_cats = [
    "N05A-Antipsychotics nervous system",
    "N05B-Anxiolytics nervous system",
    "N05C-Hypnotics and sedatives nervous system",
    "N06A-Antidepressants nervous system",
    "N06B-Psychostimulants, agents used for ADHD and nootropics nervous system",
    "N06C-Pscyholeptics and psychoanaleptics in combination nervous system",
    "N06D-Anti-dementia drugs nervous system",
    "N03-Antiepileptics nervous system",
]

def_hex_default = "lightgrey"
highlight_color = "grey"

colors = [
    highlight_color if cat in special_cats else def_hex_default
    for cat in med_cat_counts.index
]

med_cat_labels = [textwrap.fill(name, width=25) for name in med_cat_counts.index]
x_mc = np.arange(len(med_cat_counts))
fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.bar(x_mc, med_cat_counts.values, width=0.8, color=colors)
ax.set_xticks(x_mc)
ax.set_xticklabels(med_cat_labels, rotation=90)
ax.set_xlabel("Medication Category (ATC Group)")
ax.set_ylabel("Unique Opphold ID Count")
ax.bar_label(
    bars,
    labels=[f"{v:,} ({v/821*100:.2f}%)" for v in med_cat_counts.values],
    padding=2,
    fontsize=8,
)
plt.tight_layout()
plt.savefig(output_plot + "axis2/axis2_1(b)part2.png", dpi=400, bbox_inches="tight")
plt.show()

**Step 3: I check what are the most frequent diagnosis and medication based on the contact of pasient , display the top 20 then i categorize them in `Axis 3` category and display the distribution of category**

In [ ]:
import os
import sys

parent_dir = os.path.join(os.path.dirname(os.path.abspath("")))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from importall import *
from updated_mapped_icd_codes import mapped_icd_codes

output_plot = os.getenv("OUTPUT_PLOTS")

with open(os.getenv("ATCNAME_CODE_NORSK") + "atcnavn_code.json", "r") as f:
    atc_mapping = json.load(f)
atc_map = {code: info.get("atcnavn", "Unknown") for code, info in atc_mapping.items()}


input_path = os.getenv("AXISIII_MAIN_DATASET")
df = dd.read_parquet(input_path, engine="pyarrow")
print(
    "Axis 3 Unique patients • all of the above + medicated:",
    df["pasient_nr"].nunique().compute(),
)
print(
    "Unique episode of care • all of the above + medicated:",
    df["sak_nr"].nunique().compute(),
)
print(
    "Unique contacts • all of the above + medicated:",
    df["opphold_id"].nunique().compute(),
)
all_diag_codes = df["diag_diagnose"].dropna().unique().compute()
all_diag_codes = sorted(all_diag_codes)
print("All diagnosis codes in Axis 3:")
for code in all_diag_codes:
    print(f"  {code}")

diag_counts = (
    df.groupby("diag_diagnose")["opphold_id"]
    .nunique()
    .compute()
    .sort_values(ascending=False)
    .head(20)
)

diag_labels = [
    f"{code} – {mapped_icd_codes.get(code, 'Unknown')}" for code in diag_counts.index
]
wrapped_diag_labels = [textwrap.fill(lbl, width=30) for lbl in diag_labels]
width = 0.6
x_diag = np.arange(len(diag_counts))
fig, ax = plt.subplots(figsize=(16, 8))
bars = ax.bar(x_diag, diag_counts.values, width=width, color="grey", alpha=0.7)
ax.set_xticks(x_diag)
ax.set_xticklabels(wrapped_diag_labels, rotation=90)
ax.set_xlabel("Diagnosis (ICD-10 code – description)")
ax.set_ylabel("Unique Opphold ID Count")
ax.set_title("Axis 3 – Top 20 Primary Diagnoses by Opphold ID")
ax.bar_label(bars, labels=[f"{v:,}" for v in diag_counts.values], padding=2, fontsize=8)
plt.tight_layout()
plt.show()

med_counts = (
    df.groupby("atckode")["opphold_id"]
    .nunique()
    .compute()
    .sort_values(ascending=False)
    .head(20)
)

med_labels = [f"{code} – {atc_map.get(code, 'Unknown')}" for code in med_counts.index]
wrapped_med_labels = [textwrap.fill(lbl, width=30) for lbl in med_labels]
x_med = np.arange(len(med_counts))
fig, ax = plt.subplots(figsize=(16, 8))
bars = ax.bar(x_med, med_counts.values, width=width, color="#D55E00", alpha=0.6)
ax.set_xticks(x_med)
ax.set_xticklabels(wrapped_med_labels, rotation=90)
ax.set_xlabel("Medication (ATC code – name)")
ax.set_ylabel("Unique Opphold ID Count")
ax.set_title("Axis 3 – Top 20 Medications by Opphold ID")
ax.bar_label(bars, labels=[f"{v:,}" for v in med_counts.values], padding=2, fontsize=8)
plt.tight_layout()
plt.show()

categories = {
    "F70\nMild mental retardation": lambda c: isinstance(c, str) and c[:3] == "F70",
    "F71\nModerate mental retardation": lambda c: isinstance(c, str) and c[:3] == "F71",
    "F72\nSevere mental retardation": lambda c: isinstance(c, str) and c[:3] == "F72",
    "F73\nProfound mental retardation": lambda c: isinstance(c, str) and c[:3] == "F73",
    "F78\nOther mental retardation": lambda c: isinstance(c, str) and c[:3] == "F78",
    "F79\nUnspecified mental retardation": lambda c: isinstance(c, str)
    and c[:3] == "F79",
}


def map_diag_category(code: str) -> str:
    if not isinstance(code, str):
        return "Other/Unclassified"
    for cat_name, cond in categories.items():
        try:
            if cond(code):
                return cat_name
        except Exception:
            continue
    return "Other/Unclassified"


_df_diag_cat = df[["diag_diagnose", "opphold_id"]].dropna(subset=["diag_diagnose"])
_df_diag_cat = _df_diag_cat.assign(
    diag_cat=_df_diag_cat["diag_diagnose"].map(
        map_diag_category, meta=("diag_cat", "object")
    )
)

diag_cat_counts = (
    _df_diag_cat.groupby("diag_cat")["opphold_id"]
    .nunique()
    .compute()
    .sort_values(ascending=False)
)

print("\nDiagnosis category → unique opphold_id counts:")
for cat, cnt in diag_cat_counts.items():
    print(f"  {cat}: {cnt:,}")

diag_cat_labels = [textwrap.fill(name, width=25) for name in diag_cat_counts.index]
x_dc = np.arange(len(diag_cat_counts))
fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.bar(x_dc, diag_cat_counts.values, width=0.8, color="grey", alpha=0.7)
ax.set_xticks(x_dc)
ax.set_xticklabels(diag_cat_labels, rotation=90)
ax.set_xlabel("Diagnosis Category")
ax.set_ylabel("Unique Opphold ID Count")
ax.bar_label(
    bars,
    labels=[f"{v:,} ({v/65*100:.2f}%)" for v in diag_cat_counts.values],
    padding=2,
    fontsize=8,
)
plt.tight_layout()
plt.savefig(output_plot + "axis3/axis3_1(b)part1.png", dpi=400, bbox_inches="tight")
plt.show()

med_letter_map = {
    "A": "A-Alimentary tract & metabolism",
    "B": "B-Blood & blood forming organs",
    "C": "C-Cardiovascular system",
    "D": "D-Dermatologicals",
    "G": "G-Genito-urinary system & sex hormones",
    "H": "H-Systemic hormonal preparations (excl. sex hormones)",
    "J": "J-Anti-infectives for systemic use",
    "M": "M-Musculo-skeletal system",
    "N": "Other nervous system (N01-N02,N04,N07)",
    "R": "R-Respiratory system",
    "S": "S-Sensory organs",
    "V": "V-Various",
}


def map_med_category(code: str) -> str:
    if not isinstance(code, str) or len(code) < 1:
        return "Other/Unclassified"
    if code.startswith("N05A"):
        return "N05A-Antipsychotics nervous system"
    if code.startswith("N05B"):
        return "N05B-Anxiolytics nervous system"
    if code.startswith("N05C"):
        return "N05C-Hypnotics and sedatives nervous system"
    if code.startswith("N06A"):
        return "N06A-Antidepressants nervous system"
    if code.startswith("N06B"):
        return (
            "N06B-Psychostimulants, agents used for ADHD and nootropics nervous system"
        )
    if code.startswith("N06C"):
        return "N06C-Pscyholeptics and psychoanaleptics in combination nervous system"
    if code.startswith("N06D"):
        return "N06D-Anti-dementia drugs nervous system"
    if code.startswith("N03"):
        return "N03-Antiepileptics nervous system"

    letter = code[0]
    return med_letter_map.get(letter, "Other/Unclassified")


_df_med_cat = df[["atckode", "opphold_id"]].dropna(subset=["atckode"])
_df_med_cat = _df_med_cat.assign(
    med_cat=_df_med_cat["atckode"].map(map_med_category, meta=("med_cat", "object"))
)

med_cat_counts = (
    _df_med_cat.groupby("med_cat")["opphold_id"]
    .nunique()
    .compute()
    .sort_values(ascending=False)
)

print("\nMedication category → unique opphold_id counts:")
for cat, cnt in med_cat_counts.items():
    print(f"  {cat}: {cnt:,}")

special_cats = [
    "N05A-Antipsychotics nervous system",
    "N05B-Anxiolytics nervous system",
    "N05C-Hypnotics and sedatives nervous system",
    "N06A-Antidepressants nervous system",
    "N06B-Psychostimulants, agents used for ADHD and nootropics nervous system",
    "N06C-Pscyholeptics and psychoanaleptics in combination nervous system",
    "N06D-Anti-dementia drugs nervous system",
    "N03-Antiepileptics nervous system",
]

def_hex_default = "lightgrey"
highlight_color = "grey"

colors = [
    highlight_color if cat in special_cats else def_hex_default
    for cat in med_cat_counts.index
]

med_cat_labels = [textwrap.fill(name, width=25) for name in med_cat_counts.index]
x_mc = np.arange(len(med_cat_counts))
fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.bar(x_mc, med_cat_counts.values, width=0.8, color=colors)
ax.set_xticks(x_mc)
ax.set_xticklabels(med_cat_labels, rotation=90)
ax.set_xlabel("Medication Category (ATC Group)")
ax.set_ylabel("Unique Opphold ID Count")
ax.bar_label(
    bars,
    labels=[f"{v:,} ({v/65*100:.2f}%)" for v in med_cat_counts.values],
    padding=2,
    fontsize=8,
)
plt.tight_layout()
plt.savefig(output_plot + "axis3/axis3_1(b)part2.png", dpi=400, bbox_inches="tight")
plt.show()

**Research question 2(a): What are the top 5 most frequently prescribed medications for the given primary or principal diagnoses in `Axis 1` in the BUP data, considering only the given primary or principal diagnoses given to patients?**

**Research question 2(b): What are the top 5 primary or principal diagnoses associated with a corresponding prescribed medication in `Axis 1` in the BUP data, considering only the given primary or principal diagnoses given to patients?**

In [ ]:
import os
import sys

parent_dir = os.path.join(os.path.dirname(os.path.abspath("")))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from importall import *
from updated_mapped_icd_codes import mapped_icd_codes

output_plot = os.getenv("OUTPUT_PLOTS")

INPUT_PARQUET = os.getenv("AXIS1_MAIN_DATASET")

with open(os.getenv("ATCNAME_CODE_NORSK") + "atcnavn_code.json", "r") as f:
    atc_json = json.load(f)
atc_map = {code: info.get("atcnavn", "Unknown") for code, info in atc_json.items()}
cols = ["diag_diagnose", "atckode", "pasient_nr", "opphold_id"]
df = dd.read_parquet(INPUT_PARQUET, engine="pyarrow", columns=cols).compute()
diag_med_counts = (
    df.groupby(["diag_diagnose", "atckode"])["opphold_id"]
    .nunique()
    .rename("contact_count")
    .reset_index()
)
top_med_per_diag = (
    diag_med_counts.sort_values(
        ["diag_diagnose", "contact_count"], ascending=[True, False]
    )
    .groupby("diag_diagnose")
    .head(123)
    .copy()
)
top_med_per_diag["rank"] = top_med_per_diag.groupby("diag_diagnose").cumcount() + 1

rows = []
for diag, grp in top_med_per_diag.groupby("diag_diagnose"):
    rec = {"diag_diagnose": diag, "diag_name": mapped_icd_codes.get(diag, "Unknown")}
    for _, r in grp.sort_values("rank").iterrows():
        k = int(r["rank"])
        rec[f"rank{k}_med_code"] = r["atckode"]
        rec[f"rank{k}_med_name"] = atc_map.get(r["atckode"], "Unknown")
        rec[f"rank{k}_contact_count"] = int(r["contact_count"])
    rows.append(rec)

df_excel_diag = pd.DataFrame(rows).sort_values("rank1_contact_count", ascending=False)
top_diag_per_med = (
    diag_med_counts.sort_values(["atckode", "contact_count"], ascending=[True, False])
    .groupby("atckode")
    .head(123)
    .copy()
)
top_diag_per_med["rank"] = top_diag_per_med.groupby("atckode").cumcount() + 1

rows = []
for med, grp in top_diag_per_med.groupby("atckode"):
    rec = {"atckode": med, "med_name": atc_map.get(med, "Unknown")}
    for _, r in grp.sort_values("rank").iterrows():
        k = int(r["rank"])
        rec[f"rank{k}_diag_code"] = r["diag_diagnose"]
        rec[f"rank{k}_diag_name"] = mapped_icd_codes.get(r["diag_diagnose"], "Unknown")
        rec[f"rank{k}_contact_count"] = int(r["contact_count"])
    rows.append(rec)

df_excel_med = pd.DataFrame(rows).sort_values("rank1_contact_count", ascending=False)
OUT_XLSX_2 = output_plot + "axis1_all_medications_diags_by_contact.xlsx"
df_excel_med.to_excel(OUT_XLSX_2, index=False)
print(f"Written Excel: {OUT_XLSX_2}")

**To display the `Axis 1` per total axis 1 contact which is `7179` in percentage porportion**

In [ ]:
import os
import sys

parent_dir = os.path.join(os.path.dirname(os.path.abspath("")))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from importall import *
from updated_mapped_icd_codes import mapped_icd_codes

TOTAL_AXIS1_CONTACTS = 7179
OUT_XLSX_MED = (
    os.getenv("AXES_COMORBID_EXCEL_PATH")
    + "axis1_all_medications_diags_by_contact.xlsx"
)
df = pd.read_excel(OUT_XLSX_MED, sheet_name=0, dtype={"atckode": str})
df = df.rename(columns={"atckode": "med_code", "med_name": "med_name"})

diag_code_cols = [
    col for col in df.columns if col.startswith("rank") and col.endswith("_diag_code")
]
if not diag_code_cols:
    raise KeyError("No 'rank#_diag_code' columns found in the Excel file.")
max_rank = max(int(col.replace("rank", "").split("_")[0]) for col in diag_code_cols)

melted = []
for i in range(1, max_rank + 1):
    diag_code_col = f"rank{i}_diag_code"
    diag_name_col = f"rank{i}_diag_name"
    contact_col = f"rank{i}_contact_count"
    if (
        diag_code_col in df.columns
        and diag_name_col in df.columns
        and contact_col in df.columns
    ):
        temp = df[
            ["med_code", "med_name", diag_code_col, diag_name_col, contact_col]
        ].copy()
        temp = temp.rename(
            columns={
                diag_code_col: "diag_diagnose",
                diag_name_col: "diag_name",
                contact_col: "count",
            }
        )
        melted.append(temp)

long = pd.concat(melted, ignore_index=True).dropna(subset=["diag_diagnose"])

categories = {
    "F03–F09\nOrganic, including symptomatic, mental disorders": lambda c: c[
        :3
    ].startswith("F0")
    and "F03" <= c[:3] <= "F09",
    "F10–F19\nMental disorders and behavioral disorders due to the use of psychoactive substances": lambda c: "F10"
    <= c[:3]
    <= "F19",
    "F20–F29\nSchizophrenia, schizotypal disorder and paranoid disorders": lambda c: "F20"
    <= c[:3]
    <= "F29",
    "F30–F39\nAffective disorders mood disorders": lambda c: "F30" <= c[:3] <= "F39",
    "F40–F48\nNeurotic, stress-related and somatoform disorders": lambda c: "F40"
    <= c[:3]
    <= "F48",
    "F50–F59\nBehavioral syndromes associated with physiological disorders and physical factors": lambda c: "F50"
    <= c[:3]
    <= "F59",
    "F60–F69\nPersonality and behavioral disorders in adults": lambda c: "F60"
    <= c[:3]
    <= "F69",
    "F84\nPervasive developmental disorders": lambda c: c[:3].startswith("F84"),
    "F90–F98\nBehavioral and emotional disorders that usually occur in childhood and adolescence": lambda c: "F90"
    <= c[:3]
    <= "F98",
    "F99–F99\nUnspecified mental disorder": lambda c: c[:3].startswith("F99"),
    "R40–R46\nSymptoms and signs related to cognition, perception, emotional state and behavior": lambda c: "R40"
    <= c[:3]
    <= "R46",
    "Z00–Z71-PHBU\nFactors that affect health status and contact with health services": lambda c: c[
        :3
    ].startswith(
        "Z"
    )
    and "Z00" <= c[:3] <= "Z71",
    "X6n\nExternal causes/self-harm": lambda c: c[:3].startswith("X6"),
    "F710\nModerate mental retardation": lambda c: c.replace(".", "")[:3] == "F710",
}


def categorize_psych(med_code: str) -> str:
    med_code = med_code.upper()
    if med_code.startswith("N") and len(med_code) >= 4:
        prefix3 = med_code[:4]
        if prefix3 == "N05A":
            return "N05A-Antipsychotics nervous system"
        if prefix3 == "N05B":
            return "N05B-Anxiolytics nervous system"
        if prefix3 == "N05C":
            return "N05C-Hypnotics and sedatives nervous system"
        if prefix3 == "N06A":
            return "N06A-Antidepressants nervous system"
        if prefix3 == "N06B":
            return "N06B-Psychostimulants, agents used for ADHD and nootropics nervous system"
        if prefix3 == "N06C":
            return (
                "N06C-Pscyholeptics and psychoanaleptics in combination nervous system"
            )
        if prefix3 == "N06D":
            return "N06D-Anti-dementia drugs nervous system"
        if med_code.startswith("N03"):
            return "N03-Antiepileptics nervous system"
        return "Other Non-Psychotropic"
    return "Other Non-Psychotropic"


grouped_results = {}

for cat_name, selector in categories.items():
    sub = long[long["diag_diagnose"].apply(selector)].copy()
    sub["MedicationCategory"] = sub["med_code"].apply(categorize_psych)
    agg_all = (
        sub.groupby(["MedicationCategory", "med_code", "med_name"], as_index=False)[
            "count"
        ]
        .sum()
        .rename(columns={"count": "TotalContactCount"})
    )

    for med_cat, df_group in agg_all.groupby("MedicationCategory"):
        df_group = df_group.sort_values("TotalContactCount", ascending=False)
        formatted_list = []
        for _, row in df_group.iterrows():
            percent = (row["TotalContactCount"] / TOTAL_AXIS1_CONTACTS) * 100
            formatted_list.append(
                f"{row['med_code']} {row['med_name']} ({percent:.2f}%)"
            )
        grouped_results[(cat_name, med_cat)] = formatted_list

max_medications = max(len(med_list) for med_list in grouped_results.values())

rows = []
for (cat_name, med_cat), med_list in grouped_results.items():
    row = {
        "DiagnosisCategory": cat_name,
        "MedicationCategory": med_cat,
    }

    for idx, entry in enumerate(med_list, start=1):
        row[f"MedicationRank{idx}(%)"] = entry
    for idx in range(len(med_list) + 1, max_medications + 1):
        row[f"MedicationRank{idx}(%)"] = ""
    rows.append(row)

column_order = ["DiagnosisCategory", "MedicationCategory"] + [
    f"MedicationRank{i}(%)" for i in range(1, max_medications + 1)
]

result = pd.DataFrame(rows)[column_order]
OUT_XLSX_RESULT = (
    os.getenv("AXES_COMORBID_EXCEL_PATH") + "ratio/11med_diag_atc_ratio.xlsx"
)
result.to_excel(OUT_XLSX_RESULT, index=False)
print(f"Saved full table to: {OUT_XLSX_RESULT}")

**Research question 2(a): What are the top 5 most frequently prescribed medications for the given primary or principal diagnoses in `Axis 2` in the BUP data, considering only the given primary or principal diagnoses given to patients?**

**Research question 2(b): What are the top 5 primary or principal diagnoses associated with a corresponding prescribed medication in `Axis 2` in the BUP data, considering only the given primary or principal diagnoses given to patients?**

In [ ]:
import os
import sys

parent_dir = os.path.join(os.path.dirname(os.path.abspath("")))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from importall import *
from updated_mapped_icd_codes import mapped_icd_codes

output_plot = os.getenv("AXES_COMORBID_EXCEL_PATH")
INPUT_PARQUET = os.getenv("AXISII_MAIN_DATASET")

with open(os.getenv("ATCNAME_CODE_NORSK") + "atcnavn_code.json", "r") as f:
    atc_json = json.load(f)
atc_map = {code: info.get("atcnavn", "Unknown") for code, info in atc_json.items()}
cols = ["diag_diagnose", "atckode", "pasient_nr", "opphold_id"]
df = dd.read_parquet(INPUT_PARQUET, engine="pyarrow", columns=cols).compute()
diag_med_counts = (
    df.groupby(["diag_diagnose", "atckode"])["opphold_id"]
    .nunique()
    .rename("contact_count")
    .reset_index()
)
top_med_per_diag = (
    diag_med_counts.sort_values(
        ["diag_diagnose", "contact_count"], ascending=[True, False]
    )
    .groupby("diag_diagnose")
    .head(123)
    .copy()
)
top_med_per_diag["rank"] = top_med_per_diag.groupby("diag_diagnose").cumcount() + 1

rows = []
for diag, grp in top_med_per_diag.groupby("diag_diagnose"):
    rec = {"diag_diagnose": diag, "diag_name": mapped_icd_codes.get(diag, "Unknown")}
    for _, r in grp.sort_values("rank").iterrows():
        k = int(r["rank"])
        rec[f"rank{k}_med_code"] = r["atckode"]
        rec[f"rank{k}_med_name"] = atc_map.get(r["atckode"], "Unknown")
        rec[f"rank{k}_contact_count"] = int(r["contact_count"])
    rows.append(rec)

df_excel_diag = pd.DataFrame(rows).sort_values("rank1_contact_count", ascending=False)
top_diag_per_med = (
    diag_med_counts.sort_values(["atckode", "contact_count"], ascending=[True, False])
    .groupby("atckode")
    .head(123)
    .copy()
)
top_diag_per_med["rank"] = top_diag_per_med.groupby("atckode").cumcount() + 1

rows = []
for med, grp in top_diag_per_med.groupby("atckode"):
    rec = {"atckode": med, "med_name": atc_map.get(med, "Unknown")}
    for _, r in grp.sort_values("rank").iterrows():
        k = int(r["rank"])
        rec[f"rank{k}_diag_code"] = r["diag_diagnose"]
        rec[f"rank{k}_diag_name"] = mapped_icd_codes.get(r["diag_diagnose"], "Unknown")
        rec[f"rank{k}_contact_count"] = int(r["contact_count"])
    rows.append(rec)

df_excel_med = pd.DataFrame(rows).sort_values("rank1_contact_count", ascending=False)
OUT_XLSX_2 = output_plot + "axis2_all_medications_diags_by_contact.xlsx"
df_excel_med.to_excel(OUT_XLSX_2, index=False)
print(f"Written Excel: {OUT_XLSX_2}")

**To display the `Axis 2` per total axis 2 contact which are 821 in percentage porportion**

In [ ]:
import os
import sys

parent_dir = os.path.join(os.path.dirname(os.path.abspath("")))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from importall import *
from updated_mapped_icd_codes import mapped_icd_codes

TOTAL_AXIS2_CONTACTS = 821
OUT_XLSX_MED = (
    os.getenv("AXES_COMORBID_EXCEL_PATH")
    + "axis2_all_medications_diags_by_contact.xlsx"
)
df = pd.read_excel(OUT_XLSX_MED, sheet_name=0, dtype={"atckode": str})

df = df.rename(columns={"atckode": "med_code", "med_name": "med_name"})
diag_code_cols = [
    col for col in df.columns if col.startswith("rank") and col.endswith("_diag_code")
]
if not diag_code_cols:
    raise KeyError("No 'rank#_diag_code' columns found in the Excel file.")
max_rank = max(int(col.replace("rank", "").split("_")[0]) for col in diag_code_cols)

melted = []
for i in range(1, max_rank + 1):
    diag_code_col = f"rank{i}_diag_code"
    diag_name_col = f"rank{i}_diag_name"
    contact_col = f"rank{i}_contact_count"
    if (
        diag_code_col in df.columns
        and diag_name_col in df.columns
        and contact_col in df.columns
    ):
        temp = df[
            ["med_code", "med_name", diag_code_col, diag_name_col, contact_col]
        ].copy()
        temp = temp.rename(
            columns={
                diag_code_col: "diag_diagnose",
                diag_name_col: "diag_name",
                contact_col: "count",
            }
        )
        melted.append(temp)

long = pd.concat(melted, ignore_index=True).dropna(subset=["diag_diagnose"])

categories = {
    "F80\nSpecific developmental disorders of speech and language": lambda c: c.replace(
        ".", ""
    ).startswith("F80"),
    "F81\nSpecific developmental disorders of school skills, learning disabilities": lambda c: c.replace(
        ".", ""
    ).startswith(
        "F81"
    ),
    "F82\nSpecific developmental disorder in motor skills": lambda c: c.replace(
        ".", ""
    ).startswith("F82"),
    "F83\nMixed developmental disorder in specific skills": lambda c: c.replace(
        ".", ""
    ).startswith("F83"),
    "F88\nOther disorder of psychological development": lambda c: c.replace(
        ".", ""
    ).startswith("F88"),
    "F89\nUnspecified disorder of psychological development": lambda c: c.replace(
        ".", ""
    ).startswith("F89"),
}


def categorize_psych(med_code: str) -> str:

    med_code = med_code.upper()
    if med_code.startswith("N") and len(med_code) >= 4:
        prefix3 = med_code[:4]
        if prefix3 == "N05A":
            return "N05A-Antipsychotics nervous system"
        if prefix3 == "N05B":
            return "N05B-Anxiolytics nervous system"
        if prefix3 == "N05C":
            return "N05C-Hypnotics and sedatives nervous system"
        if prefix3 == "N06A":
            return "N06A-Antidepressants nervous system"
        if prefix3 == "N06B":
            return "N06B-Psychostimulants, agents used for ADHD and nootropics nervous system"
        if prefix3 == "N06C":
            return (
                "N06C-Pscyholeptics and psychoanaleptics in combination nervous system"
            )
        if prefix3 == "N06D":
            return "N06D-Anti-dementia drugs nervous system"
        if med_code.startswith("N03"):
            return "N03-Antiepileptics nervous system"
        return "Other Non-Psychotropic"
    return "Other Non-Psychotropic"


grouped_results = {}

for cat_name, selector in categories.items():
    sub = long[long["diag_diagnose"].apply(selector)].copy()
    sub["MedicationCategory"] = sub["med_code"].apply(categorize_psych)
    agg_all = (
        sub.groupby(["MedicationCategory", "med_code", "med_name"], as_index=False)[
            "count"
        ]
        .sum()
        .rename(columns={"count": "TotalContactCount"})
    )
    for med_cat, df_group in agg_all.groupby("MedicationCategory"):
        df_group = df_group.sort_values("TotalContactCount", ascending=False)
        formatted_list = []
        for _, row in df_group.iterrows():
            percent = (row["TotalContactCount"] / TOTAL_AXIS2_CONTACTS) * 100
            formatted_list.append(
                f"{row['med_code']} {row['med_name']} ({percent:.1f}%)"
            )
        grouped_results[(cat_name, med_cat)] = formatted_list
max_medications = max(len(med_list) for med_list in grouped_results.values())

rows = []
for (cat_name, med_cat), med_list in grouped_results.items():
    row = {
        "DiagnosisCategory": cat_name,
        "MedicationCategory": med_cat,
    }

    for idx, entry in enumerate(med_list, start=1):
        row[f"MedicationRank{idx}(%)"] = entry
    for idx in range(len(med_list) + 1, max_medications + 1):
        row[f"MedicationRank{idx}(%)"] = ""
    rows.append(row)

column_order = ["DiagnosisCategory", "MedicationCategory"] + [
    f"MedicationRank{i}(%)" for i in range(1, max_medications + 1)
]

result = pd.DataFrame(rows)[column_order]

OUT_XLSX_RESULT = os.getenv("AXES_COMORBID_EXCEL_PATH") + "22med_diag_atc_ratio.xlsx"
result.to_excel(OUT_XLSX_RESULT, index=False)

print(f"Saved full table to: {OUT_XLSX_RESULT}")

**Research question 2(a): What are the top 5 most frequently prescribed medications for the given primary or principal diagnoses in `Axis 3` in the BUP data, considering only the given primary or principal diagnoses given to patients?**

**Research question 2(b): What are the top 5 primary or principal diagnoses associated with a corresponding prescribed medication in `Axis 3` in the BUP data, considering only the given primary or principal diagnoses given to patients?**

In [ ]:
import os
import sys

parent_dir = os.path.join(os.path.dirname(os.path.abspath("")))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from importall import *
from updated_mapped_icd_codes import mapped_icd_codes

output_plot = "/home/dipendrp/workbench/Paper4_v3/P4_V3/P4_V4/"
INPUT_PARQUET = (
    "/home/dipendrp/workbench/Paper4_v3/P4_V3/data/v2axis3_primary_medicated"
)

with open("/home/dipendrp/workbench/Paper4_v3/P4_V3/atcnavn_code.json", "r") as f:
    atc_json = json.load(f)
atc_map = {code: info.get("atcnavn", "Unknown") for code, info in atc_json.items()}
cols = ["diag_diagnose", "atckode", "pasient_nr", "opphold_id"]
df = dd.read_parquet(INPUT_PARQUET, engine="pyarrow", columns=cols).compute()
diag_med_counts = (
    df.groupby(["diag_diagnose", "atckode"])["opphold_id"]
    .nunique()
    .rename("contact_count")
    .reset_index()
)
top_med_per_diag = (
    diag_med_counts.sort_values(
        ["diag_diagnose", "contact_count"], ascending=[True, False]
    )
    .groupby("diag_diagnose")
    .head(123)
    .copy()
)
top_med_per_diag["rank"] = top_med_per_diag.groupby("diag_diagnose").cumcount() + 1

rows = []
for diag, grp in top_med_per_diag.groupby("diag_diagnose"):
    rec = {"diag_diagnose": diag, "diag_name": mapped_icd_codes.get(diag, "Unknown")}
    for _, r in grp.sort_values("rank").iterrows():
        k = int(r["rank"])
        rec[f"rank{k}_med_code"] = r["atckode"]
        rec[f"rank{k}_med_name"] = atc_map.get(r["atckode"], "Unknown")
        rec[f"rank{k}_contact_count"] = int(r["contact_count"])
    rows.append(rec)

df_excel_diag = pd.DataFrame(rows).sort_values("rank1_contact_count", ascending=False)
top_diag_per_med = (
    diag_med_counts.sort_values(["atckode", "contact_count"], ascending=[True, False])
    .groupby("atckode")
    .head(123)
    .copy()
)
top_diag_per_med["rank"] = top_diag_per_med.groupby("atckode").cumcount() + 1

rows = []
for med, grp in top_diag_per_med.groupby("atckode"):
    rec = {"atckode": med, "med_name": atc_map.get(med, "Unknown")}
    for _, r in grp.sort_values("rank").iterrows():
        k = int(r["rank"])
        rec[f"rank{k}_diag_code"] = r["diag_diagnose"]
        rec[f"rank{k}_diag_name"] = mapped_icd_codes.get(r["diag_diagnose"], "Unknown")
        rec[f"rank{k}_contact_count"] = int(r["contact_count"])
    rows.append(rec)

df_excel_med = pd.DataFrame(rows).sort_values("rank1_contact_count", ascending=False)
OUT_XLSX_2 = output_plot + "axis3_all_medications_diags_by_contact.xlsx"
df_excel_med.to_excel(OUT_XLSX_2, index=False)
print(f"Written Excel: {OUT_XLSX_2}")

**To display the `Axis 3` per total axis 3 contact - 65 percentage porportion**

In [ ]:
import os
import sys

parent_dir = os.path.join(os.path.dirname(os.path.abspath("")))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from importall import *
from updated_mapped_icd_codes import mapped_icd_codes

TOTAL_AXIS3_CONTACTS = 65

OUT_XLSX_MED = (
    os.getenv("AXES_COMORBID_EXCEL_PATH")
    + "axis3_all_medications_diags_by_contact.xlsx"
)
df = pd.read_excel(OUT_XLSX_MED, sheet_name=0, dtype={"atckode": str})
df = df.rename(columns={"atckode": "med_code", "med_name": "med_name"})
diag_code_cols = [
    col for col in df.columns if col.startswith("rank") and col.endswith("_diag_code")
]
if not diag_code_cols:
    raise KeyError("No 'rank#_diag_code' columns found in the Excel file.")
max_rank = max(int(col.replace("rank", "").split("_")[0]) for col in diag_code_cols)


melted = []
for i in range(1, max_rank + 1):
    diag_code_col = f"rank{i}_diag_code"
    diag_name_col = f"rank{i}_diag_name"
    contact_col = f"rank{i}_contact_count"
    if (
        diag_code_col in df.columns
        and diag_name_col in df.columns
        and contact_col in df.columns
    ):
        temp = df[
            ["med_code", "med_name", diag_code_col, diag_name_col, contact_col]
        ].copy()
        temp = temp.rename(
            columns={
                diag_code_col: "diag_diagnose",
                diag_name_col: "diag_name",
                contact_col: "count",
            }
        )
        melted.append(temp)

long = pd.concat(melted, ignore_index=True).dropna(subset=["diag_diagnose"])

categories = {
    "F70\nMild mental retardation": lambda c: c.startswith("F70"),
    "F71\nModerate mental retardation": lambda c: c.startswith("F71"),
    "F72\nSevere mental retardation": lambda c: c.startswith("F72"),
    "F73\nProfound mental retardation": lambda c: c.startswith("F73"),
    "F78\nOther mental retardation": lambda c: c.startswith("F78"),
    "F79\nUnspecified mental retardation": lambda c: c.startswith("F79"),
}


def categorize_psych(med_code: str) -> str:

    med_code = med_code.upper()
    if med_code.startswith("N") and len(med_code) >= 4:
        prefix3 = med_code[:4]
        if prefix3 == "N05A":
            return "N05A-Antipsychotics nervous system"
        if prefix3 == "N05B":
            return "N05B-Anxiolytics nervous system"
        if prefix3 == "N05C":
            return "N05C-Hypnotics and sedatives nervous system"
        if prefix3 == "N06A":
            return "N06A-Antidepressants nervous system"
        if prefix3 == "N06B":
            return "N06B-Psychostimulants, agents used for ADHD and nootropics nervous system"
        if prefix3 == "N06C":
            return (
                "N06C-Pscyholeptics and psychoanaleptics in combination nervous system"
            )
        if prefix3 == "N06D":
            return "N06D-Anti-dementia drugs nervous system"
        if med_code.startswith("N03"):
            return "N03-Antiepileptics nervous system"
        return "Other Non-Psychotropic"
    return "Other Non-Psychotropic"


grouped_results = {}

for cat_name, selector in categories.items():
    sub = long[long["diag_diagnose"].apply(selector)].copy()
    sub["MedicationCategory"] = sub["med_code"].apply(categorize_psych)

    agg_all = (
        sub.groupby(["MedicationCategory", "med_code", "med_name"], as_index=False)[
            "count"
        ]
        .sum()
        .rename(columns={"count": "TotalContactCount"})
    )

    for med_cat, df_group in agg_all.groupby("MedicationCategory"):
        df_group = df_group.sort_values("TotalContactCount", ascending=False)
        formatted_list = []
        for _, row in df_group.iterrows():
            percent = (row["TotalContactCount"] / TOTAL_AXIS3_CONTACTS) * 100
            formatted_list.append(
                f"{row['med_code']} {row['med_name']} ({percent:.1f}%)"
            )
        grouped_results[(cat_name, med_cat)] = formatted_list

max_medications = max(len(med_list) for med_list in grouped_results.values())

rows = []
for (cat_name, med_cat), med_list in grouped_results.items():
    row = {
        "DiagnosisCategory": cat_name,
        "MedicationCategory": med_cat,
    }
    for idx, entry in enumerate(med_list, start=1):
        row[f"MedicationRank{idx}(%)"] = entry
    for idx in range(len(med_list) + 1, max_medications + 1):
        row[f"MedicationRank{idx}(%)"] = ""
    rows.append(row)


column_order = ["DiagnosisCategory", "MedicationCategory"] + [
    f"MedicationRank{i}(%)" for i in range(1, max_medications + 1)
]

result = pd.DataFrame(rows)[column_order]

OUT_XLSX_RESULT = os.getenv("AXES_COMORBID_EXCEL_PATH") + "33med_diag_atc_ratio.xlsx"
result.to_excel(OUT_XLSX_RESULT, index=False)

print(f"Saved full table to: {OUT_XLSX_RESULT}")